# Recreate KaroSpace `Features > Markers`

This notebook dissects the calculation behind the marker list shown in `Features > Markers`.

The calculation is expanded here so each step can be inspected: expression matrix selection, normalization, count filtering, category retention, Wilcoxon rank-sum testing, multiple-testing correction, log2 fold change calculation, and the final marker list selection.

## Calculation Summary

For each categorical annotation column and each category:

1. Resolve the analysis matrix. If `wilcoxon_layer` is not set, KaroSpace uses `layers['normalized']` as-is when present; otherwise it uses `layers['counts']` when present, falling back to `adata.X`, and applies library-size normalization to `target_sum = 10000` followed by `log1p`.
2. Resolve the count-filter matrix. KaroSpace uses `statistics_counts_layer` when present, otherwise `adata.X`.
3. Drop cells with total counts below `statistics_min_cell_counts`.
4. Drop features with total counts below `statistics_min_feature_counts` after the cell filter.
5. Keep categories with at least `wilcoxon_min_cells_per_group` cells.
6. For each retained category, compare that category against all other retained categories (`one-vs-rest`).
7. For each feature, compute a Wilcoxon rank-sum p-value and score. KaroSpace's production path asks Scanpy for this table when Scanpy is available; this notebook uses the explicit SciPy rank-sum calculation so the math is visible.
8. Compute per-feature means on the Wilcoxon expression matrix and `log2FC = log2(mean_source + 1e-9) - log2(mean_rest + 1e-9)`.
9. Adjust p-values with the configured correction method.
10. Drop features below `wilcoxon_min_pct_expressed` in both groups, if that threshold is enabled.
11. For the category-vs-rest table, keep source-enriched rows with `log2FC >= wilcoxon_log2fc_cutoff`, sort by adjusted p-value, raw p-value, absolute log2FC descending, and feature name, then keep `wilcoxon_top_n_per_category` rows.
12. The marker list shown in `Features > Markers` is a second selection over that DE payload: finite `padj < wilcoxon_padj_cutoff` and `log2FC >= wilcoxon_log2fc_cutoff`, sorted by adjusted p-value, absolute log2FC descending, and feature name, capped at `marker_limit_per_category`.

In [64]:
from pathlib import Path
import json
import re

import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse, stats


## Inputs

Change these values to match the export you want to reproduce. The defaults run against the repository's bundled multimodal test dataset.

In [65]:
h5ad_path = "../tests/comet_xenium_multimodal.h5ad"

# Use "rna" for adata.X. For the bundled test data, "protein" maps to adata.obsm["protein"].
modality = "rna"
obsm_modality_var_keys = {"protein": "protein_var"}

# These should match --main-cell-annotation plus --statistics-additional-annotations from the export.
annotation_cols = ["leiden_rna"]

# Export/statistics settings.
wilcoxon_layer = None
statistics_counts_layer = "counts"
statistics_normalization = "RC"
statistics_scale_factor = 10000.0
statistics_normalized_layer = None

statistics_min_cell_counts = 10
statistics_min_feature_counts = 10
wilcoxon_min_cells_per_group = 10
wilcoxon_min_pct_expressed = 0.2
wilcoxon_p_adjust_method = "fdr_bh"
wilcoxon_padj_cutoff = 0.05
wilcoxon_log2fc_cutoff = 1.0
wilcoxon_top_n_per_category = 300

# Export-time cap used for the Features > Markers list.
marker_limit_per_category = 50

# Smaller chunks use less memory but take longer.
wilcoxon_feature_chunk_size = 128


## Matrix And Normalization Helpers

In [66]:
WILCOXON_TARGET_SUM = 10000.0
LOG2FC_EPSILON = 1e-9


def copy_matrix(matrix):
    if sparse.issparse(matrix):
        return matrix.copy()
    return np.array(matrix, dtype=float, copy=True)


def matrix_axis_sum(matrix, axis):
    values = np.asarray(matrix.sum(axis=axis)).ravel() if sparse.issparse(matrix) else np.asarray(matrix, dtype=float).sum(axis=axis)
    values = np.asarray(values, dtype=float).ravel()
    values[~np.isfinite(values)] = 0.0
    return values


def library_size_normalize(matrix, target_sum=WILCOXON_TARGET_SUM):
    matrix = copy_matrix(matrix)
    target_sum = float(target_sum)
    if sparse.issparse(matrix):
        matrix = matrix.astype(np.float64, copy=False).tocsr()
        matrix.data[~np.isfinite(matrix.data)] = 0.0
        row_sums = np.asarray(matrix.sum(axis=1)).ravel()
        scale = np.divide(target_sum, row_sums, out=np.zeros_like(row_sums, dtype=float), where=row_sums > 0)
        normalized = sparse.diags(scale).dot(matrix).tocsr()
        normalized.data[~np.isfinite(normalized.data)] = 0.0
        normalized.eliminate_zeros()
        return normalized

    matrix[~np.isfinite(matrix)] = 0.0
    row_sums = matrix.sum(axis=1)
    scale = np.divide(target_sum, row_sums, out=np.zeros_like(row_sums, dtype=float), where=row_sums > 0)
    matrix *= scale[:, None]
    matrix[~np.isfinite(matrix)] = 0.0
    return matrix


def log_normalize(matrix, target_sum=WILCOXON_TARGET_SUM):
    normalized = library_size_normalize(matrix, target_sum=target_sum)
    if sparse.issparse(normalized):
        normalized = normalized.tocsr(copy=True)
        normalized.data = np.log1p(normalized.data)
        normalized.data[~np.isfinite(normalized.data)] = 0.0
        normalized.eliminate_zeros()
        return normalized
    normalized = np.log1p(normalized)
    normalized[~np.isfinite(normalized)] = 0.0
    return normalized


def matrix_to_dense(matrix):
    return matrix.toarray() if sparse.issparse(matrix) else np.asarray(matrix, dtype=float)


def subset_matrix(matrix, row_mask, feature_mask):
    subset = matrix[row_mask]
    subset = subset[:, feature_mask]
    return subset.copy() if sparse.issparse(subset) else np.asarray(subset, dtype=float)


def materialize_rows(matrix, row_mask):
    subset = matrix[row_mask]
    return subset.copy() if sparse.issparse(subset) else np.asarray(subset, dtype=float)


def column_means(matrix, row_mask):
    if not bool(np.any(row_mask)):
        return np.zeros(int(matrix.shape[1]), dtype=float)
    subset = matrix[row_mask]
    means = np.asarray(subset.mean(axis=0), dtype=float).ravel() if sparse.issparse(subset) else np.asarray(subset, dtype=float).mean(axis=0)
    means = np.asarray(means, dtype=float)
    means[~np.isfinite(means)] = 0.0
    return means


def positive_fraction(matrix, row_mask, feature_indices):
    if not bool(np.any(row_mask)):
        return [None for _ in feature_indices]
    subset = matrix[row_mask]
    if sparse.issparse(subset):
        subset = subset[:, list(feature_indices)]
        counts = np.asarray((subset > 0).sum(axis=0)).ravel()
    else:
        subset = np.asarray(subset, dtype=float)[:, list(feature_indices)]
        counts = np.count_nonzero(subset > 0, axis=0)
    return [float(value) / int(row_mask.sum()) for value in counts]


def compact_float(value, significant_digits=6):
    try:
        value = float(value)
    except (TypeError, ValueError):
        return None
    if not np.isfinite(value):
        return None
    return float(f"{value:.{max(1, int(significant_digits))}g}")


## Modality And Matrix Resolution

These cells reproduce the decisions KaroSpace makes before the statistics run. They are written out here rather than imported from the package so the branch taken by your dataset is visible.

In [67]:
def make_modality_adata(adata, modality_name, obsm_var_keys=None):
    modality_name = str(modality_name)
    obsm_var_keys = obsm_var_keys or {}
    if modality_name == "rna":
        return adata

    if modality_name not in adata.obsm:
        raise KeyError(f"{modality_name!r} is not an obsm matrix. Available: {list(adata.obsm.keys())}")

    matrix = adata.obsm[modality_name]
    var_key = obsm_var_keys.get(modality_name, f"{modality_name}_var")
    var_table = adata.uns.get(var_key)
    if isinstance(var_table, pd.DataFrame) and not var_table.empty:
        var = var_table.copy()
        var.index = var.iloc[:, 0].astype(str).to_numpy()
    else:
        var = pd.DataFrame(index=[str(i) for i in range(int(matrix.shape[1]))])

    return ad.AnnData(X=matrix, obs=adata.obs.copy(), var=var)


def resolve_wilcoxon_matrix(adata, expression_layer=None):
    layers = getattr(adata, "layers", None) or {}
    if expression_layer:
        if expression_layer in layers:
            if str(expression_layer) == "normalized":
                return layers[expression_layer], str(expression_layer)
            return log_normalize(layers[expression_layer]), f"{expression_layer}_log1p_normalized"
        print(f"Requested wilcoxon_layer={expression_layer!r} was not found; using the automatic fallback.")

    if "normalized" in layers:
        return layers["normalized"], "normalized"
    if "counts" in layers:
        return log_normalize(layers["counts"]), "counts_log1p_normalized"
    return log_normalize(adata.X), "X_log1p_normalized"


def resolve_distribution_matrix(adata, counts_layer="counts", normalization="RC", scale_factor=10000.0, normalized_layer=None):
    layers = getattr(adata, "layers", None) or {}
    if normalized_layer:
        if normalized_layer in layers:
            return layers[normalized_layer], f"{normalized_layer}_layer"
        print(f"Requested statistics_normalized_layer={normalized_layer!r} was not found; falling back to counts normalization.")

    if counts_layer and counts_layer in layers:
        source_matrix = layers[counts_layer]
        source_name = str(counts_layer)
    else:
        if counts_layer:
            print(f"statistics_counts_layer={counts_layer!r} was not found; using adata.X.")
        source_matrix = adata.X
        source_name = "X"

    method = str(normalization or "RC").strip().lower()
    if method in {"rc", "relative_counts", "relative-counts", "relativecounts"}:
        return library_size_normalize(source_matrix, target_sum=float(scale_factor)), f"{source_name}_library_normalized"
    if method in {"lognormalize", "log_normalize", "log-normalize", "lognorm"}:
        return log_normalize(source_matrix, target_sum=WILCOXON_TARGET_SUM), f"{source_name}_log_normalized"
    raise ValueError("statistics_normalization must be 'RC' or 'LogNormalize'")


def resolve_count_filter_matrix(adata, counts_layer="counts"):
    layers = getattr(adata, "layers", None) or {}
    if counts_layer and counts_layer in layers:
        return layers[counts_layer], str(counts_layer)
    if counts_layer:
        print(f"statistics_counts_layer={counts_layer!r} was not found; using adata.X for count filters.")
    return adata.X, "X"


In [68]:
adata = ad.read_h5ad(h5ad_path)
analysis_adata = make_modality_adata(adata, modality, obsm_modality_var_keys)

wilcoxon_expression_matrix, wilcoxon_expression_source = resolve_wilcoxon_matrix(analysis_adata, wilcoxon_layer)
distribution_expression_matrix, distribution_expression_source = resolve_distribution_matrix(
    analysis_adata,
    counts_layer=statistics_counts_layer,
    normalization=statistics_normalization,
    scale_factor=statistics_scale_factor,
    normalized_layer=statistics_normalized_layer,
)
statistics_filter_counts_matrix, statistics_filter_counts_source = resolve_count_filter_matrix(
    analysis_adata,
    counts_layer=statistics_counts_layer,
)

print(f"Loaded: {h5ad_path}")
print(f"Modality: {modality}")
print(f"Cells: {analysis_adata.n_obs:,}")
print(f"Features: {analysis_adata.n_vars:,}")
print(f"Wilcoxon expression source: {wilcoxon_expression_source}")
print(f"Distribution/means expression source: {distribution_expression_source}")
print(f"Statistics count-filter source: {statistics_filter_counts_source}")


statistics_counts_layer='counts' was not found; using adata.X.
statistics_counts_layer='counts' was not found; using adata.X for count filters.
Loaded: ../tests/comet_xenium_multimodal.h5ad
Modality: rna
Cells: 46,003
Features: 5,001
Wilcoxon expression source: X_log1p_normalized
Distribution/means expression source: X_library_normalized
Statistics count-filter source: X


## Statistics Helpers

This is the expanded statistical path. The p-value adjustment implementation follows the same methods supported by KaroSpace: `fdr_bh`, `bonferroni`, `holm`, and `none`.

In [69]:
def normalize_pct_threshold(value):
    threshold = float(value or 0.0)
    if threshold > 1.0:
        threshold = threshold / 100.0
    return min(max(threshold, 0.0), 1.0)


def adjust_pvalues(pvalues, method="fdr_bh"):
    p = np.asarray(pvalues, dtype=float)
    adjusted = np.full(p.shape, np.nan, dtype=float)
    finite = np.isfinite(p)
    if not finite.any():
        return adjusted

    idx = np.flatnonzero(finite)
    vals = np.clip(p[idx], 0, 1)
    method_norm = str(method or "fdr_bh").strip().lower().replace("-", "_")

    if method_norm in {"none", "raw", "pvalue", "pvalues"}:
        adjusted[idx] = vals
    elif method_norm in {"bonferroni", "bonf"}:
        adjusted[idx] = np.clip(vals * len(vals), 0, 1)
    else:
        order = np.argsort(vals)
        ranked = vals[order]
        n = len(ranked)
        if method_norm in {"holm", "holm_bonferroni"}:
            adj = (n - np.arange(n, dtype=float)) * ranked
            adj = np.maximum.accumulate(adj)
        else:
            adj = ranked * n / (np.arange(n, dtype=float) + 1.0)
            adj = np.minimum.accumulate(adj[::-1])[::-1]
        adjusted[idx[order]] = np.clip(adj, 0, 1)

    adjusted[(adjusted == 0) & np.isfinite(adjusted)] = np.nextafter(0.0, 1.0)
    return adjusted


def wilcoxon_rank_table(matrix, labels, source, reference="rest", feature_names=None, chunk_size=128):
    labels = labels.astype(str)
    source_mask = labels == str(source)
    reference_mask = ~source_mask if str(reference) == "rest" else labels == str(reference)
    pair_mask = source_mask | reference_mask
    pair_labels = labels[pair_mask]
    pair_matrix = materialize_rows(matrix, pair_mask)
    pair_source_mask = pair_labels == str(source)
    pair_reference_mask = ~pair_source_mask if str(reference) == "rest" else pair_labels == str(reference)

    n_features = int(pair_matrix.shape[1])
    if feature_names is None:
        feature_names = [str(i) for i in range(n_features)]

    rows = []
    for start in range(0, n_features, int(chunk_size)):
        end = min(start + int(chunk_size), n_features)
        block = matrix_to_dense(pair_matrix[:, start:end])
        source_block = block[pair_source_mask]
        reference_block = block[pair_reference_mask]
        if source_block.shape[0] == 0 or reference_block.shape[0] == 0:
            scores = np.zeros(end - start, dtype=float)
            pvalues = np.ones(end - start, dtype=float)
        else:
            try:
                test = stats.ranksums(source_block, reference_block, axis=0, nan_policy="propagate")
                scores = np.asarray(test.statistic, dtype=float)
                pvalues = np.asarray(test.pvalue, dtype=float)
            except TypeError:
                chunk_scores = []
                chunk_pvalues = []
                for offset in range(end - start):
                    test = stats.ranksums(source_block[:, offset], reference_block[:, offset])
                    chunk_scores.append(test.statistic)
                    chunk_pvalues.append(test.pvalue)
                scores = np.asarray(chunk_scores, dtype=float)
                pvalues = np.asarray(chunk_pvalues, dtype=float)
        scores[~np.isfinite(scores)] = 0.0
        pvalues[~np.isfinite(pvalues)] = 1.0
        pvalues = np.clip(pvalues, 0.0, 1.0)
        for offset, feature in enumerate(feature_names[start:end]):
            rows.append({"feature": str(feature), "score": float(scores[offset]), "pvalue": float(pvalues[offset])})
    return pd.DataFrame(rows)


def format_wilcoxon_result(rank_table, source_mask, reference_mask, expression_matrix, feature_names, expression_source, *, p_adjust_method, min_pct_expressed, padj_cutoff, log2fc_cutoff, top_n, log2fc_direction="positive"):
    n_source = int(source_mask.sum())
    n_reference = int(reference_mask.sum())
    min_pct = normalize_pct_threshold(min_pct_expressed)
    source_means = column_means(expression_matrix, source_mask)
    reference_means = column_means(expression_matrix, reference_mask)
    base_means = column_means(expression_matrix, source_mask | reference_mask)
    feature_to_idx = {str(feature): idx for idx, feature in enumerate(feature_names)}

    work = rank_table.copy()
    work["_feature"] = work["feature"].astype(str)
    work["_feature_idx"] = [feature_to_idx.get(feature) for feature in work["_feature"]]
    work = work[work["_feature_idx"].notna()].copy()
    if work.empty:
        return {"available": True, "features": [], "log2foldchanges": [], "pvals": [], "pvals_adj": [], "scores": [], "pct_source": [], "pct_reference": [], "base_mean": []}

    indices = [int(idx) for idx in work["_feature_idx"]]
    pvals = pd.to_numeric(work["pvalue"], errors="coerce").to_numpy(dtype=float)
    pvals[~np.isfinite(pvals)] = 1.0
    pvals = np.clip(pvals, 0.0, 1.0)
    padj = adjust_pvalues(pvals, p_adjust_method)
    padj[~np.isfinite(padj)] = 1.0

    work["_pct_source"] = [float(v) if v is not None and np.isfinite(v) else 0.0 for v in positive_fraction(expression_matrix, source_mask, indices)]
    work["_pct_reference"] = [float(v) if v is not None and np.isfinite(v) else 0.0 for v in positive_fraction(expression_matrix, reference_mask, indices)]
    work["_log2fc"] = np.log2(source_means[indices] + LOG2FC_EPSILON) - np.log2(reference_means[indices] + LOG2FC_EPSILON)
    work["_pvalue"] = pvals
    work["_padj"] = padj
    work["_base_mean"] = base_means[indices]
    work["_score"] = pd.to_numeric(work["score"], errors="coerce").fillna(0.0).to_numpy(dtype=float)

    if min_pct > 0:
        work = work[(work["_pct_source"] >= min_pct) | (work["_pct_reference"] >= min_pct)].copy()
    work = work[np.isfinite(work["_log2fc"].to_numpy(dtype=float))].copy()
    min_pct_feature_count = int(work.shape[0])

    if str(log2fc_direction or "two_sided") == "positive":
        work = work[work["_log2fc"] >= float(log2fc_cutoff)].copy()
    work["_abs_lfc"] = np.abs(work["_log2fc"].to_numpy(dtype=float))
    work = work.sort_values(["_padj", "_pvalue", "_abs_lfc", "_feature"], ascending=[True, True, False, True]).head(max(1, int(top_n)))

    return {
        "available": True,
        "method": "cell-wilcoxon-rest-expanded-notebook",
        "p_adjust_method": str(p_adjust_method or "fdr_bh").strip().lower().replace("-", "_"),
        "min_pct_expressed": min_pct,
        "padj_cutoff": float(padj_cutoff),
        "log2fc_cutoff": float(log2fc_cutoff),
        "table_top_n": int(top_n),
        "n_source": n_source,
        "n_reference": n_reference,
        "n_replicates": 0,
        "counts_layer": expression_source,
        "min_pct_feature_count": min_pct_feature_count,
        "features": work["_feature"].astype(str).tolist(),
        "log2foldchanges": [compact_float(v, 6) for v in work["_log2fc"]],
        "pvals": [compact_float(v, 6) for v in work["_pvalue"]],
        "pvals_adj": [compact_float(v, 6) for v in work["_padj"]],
        "scores": [compact_float(v, 6) for v in work["_score"]],
        "pct_source": [compact_float(v, 5) for v in work["_pct_source"]],
        "pct_reference": [compact_float(v, 5) for v in work["_pct_reference"]],
        "base_mean": [compact_float(v, 6) for v in work["_base_mean"]],
    }


## Apply Count Filters And Retain Categories

In [70]:
def prepare_annotation_inputs(adata, annotation_col, expression_matrix, summary_matrix, count_filter_matrix):
    if annotation_col not in adata.obs.columns:
        raise KeyError(f"{annotation_col!r} is not in adata.obs")
    col = adata.obs[annotation_col]
    if pd.api.types.is_numeric_dtype(col):
        raise TypeError("Features > Markers requires a categorical annotation, not a numeric column")
    if not isinstance(col.dtype, pd.CategoricalDtype):
        col = col.astype("category")

    labels = col.astype(str).to_numpy()
    categories = [str(category) for category in col.cat.categories]
    feature_names = [str(feature) for feature in adata.var_names]

    cell_mask = np.ones(int(labels.shape[0]), dtype=bool)
    if int(statistics_min_cell_counts) > 0:
        cell_totals = matrix_axis_sum(count_filter_matrix, axis=1)
        cell_mask = np.isfinite(cell_totals) & (cell_totals >= int(statistics_min_cell_counts))

    feature_mask = np.ones(len(feature_names), dtype=bool)
    if int(statistics_min_feature_counts) > 0:
        feature_totals = matrix_axis_sum(count_filter_matrix[cell_mask], axis=0)
        feature_mask = np.isfinite(feature_totals) & (feature_totals >= int(statistics_min_feature_counts))

    filtered_labels = labels[cell_mask]
    filtered_expression = subset_matrix(expression_matrix, cell_mask, feature_mask)
    filtered_summary = subset_matrix(summary_matrix, cell_mask, feature_mask)
    filtered_feature_names = [feature for feature, keep in zip(feature_names, feature_mask) if bool(keep)]
    retained_categories = [
        category
        for category in categories
        if int(np.count_nonzero(filtered_labels == category)) >= int(wilcoxon_min_cells_per_group)
    ]

    return {
        "labels": filtered_labels,
        "expression_matrix": filtered_expression,
        "summary_matrix": filtered_summary,
        "feature_names": filtered_feature_names,
        "categories": categories,
        "retained_categories": retained_categories,
        "n_cells_after_filter": int(filtered_labels.shape[0]),
        "n_features_after_filter": int(len(filtered_feature_names)),
    }


prepared = {
    annotation_col: prepare_annotation_inputs(
        analysis_adata,
        annotation_col,
        wilcoxon_expression_matrix,
        distribution_expression_matrix,
        statistics_filter_counts_matrix,
    )
    for annotation_col in annotation_cols
}

pd.DataFrame([
    {
        "annotation": key,
        "cells_after_count_filter": value["n_cells_after_filter"],
        "features_after_count_filter": value["n_features_after_filter"],
        "retained_categories": len(value["retained_categories"]),
        "retained_category_names": ", ".join(value["retained_categories"]),
    }
    for key, value in prepared.items()
])


,annotation,cells_after_count_filter,features_after_count_filter,retained_categories,retained_category_names
0,leiden_rna,44925,5001,13,"0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12"


## Compute One-Vs-Rest Wilcoxon Marker Payload

`Features > Markers` uses the category-vs-rest marker list when it exists, so this notebook focuses on that path. Pairwise category-vs-category contrasts are used elsewhere in KaroSpace's comparison panels.

In [71]:
wilcoxon_de = {}

for annotation_col, info in prepared.items():
    labels = info["labels"]
    retained_categories = info["retained_categories"]
    if len(retained_categories) < 2:
        print(f"Skipping {annotation_col}: fewer than two retained categories")
        continue

    retained_mask = np.isin(labels, retained_categories)
    retained_matrix = materialize_rows(info["expression_matrix"], retained_mask)
    retained_labels = labels[retained_mask]
    annotation_payload = {}

    for category in retained_categories:
        source_mask = labels == category
        reference_mask = np.isin(labels, [other for other in retained_categories if other != category])
        if int(source_mask.sum()) < int(wilcoxon_min_cells_per_group) or int(reference_mask.sum()) < int(wilcoxon_min_cells_per_group):
            continue

        print(f"{annotation_col}: {category} vs rest ({int(source_mask.sum()):,} vs {int(reference_mask.sum()):,} cells)")
        rank_table = wilcoxon_rank_table(
            retained_matrix,
            retained_labels,
            source=category,
            reference="rest",
            feature_names=info["feature_names"],
            chunk_size=wilcoxon_feature_chunk_size,
        )
        formatted = format_wilcoxon_result(
            rank_table,
            source_mask=source_mask,
            reference_mask=reference_mask,
            expression_matrix=info["expression_matrix"],
            feature_names=info["feature_names"],
            expression_source=wilcoxon_expression_source,
            p_adjust_method=wilcoxon_p_adjust_method,
            min_pct_expressed=wilcoxon_min_pct_expressed,
            padj_cutoff=wilcoxon_padj_cutoff,
            log2fc_cutoff=wilcoxon_log2fc_cutoff,
            top_n=wilcoxon_top_n_per_category,
            log2fc_direction="positive",
        )
        annotation_payload.setdefault(category, {})["__rest__"] = formatted

    if annotation_payload:
        wilcoxon_de[annotation_col] = annotation_payload

print(f"Computed expanded Wilcoxon payloads for: {list(wilcoxon_de)}")


leiden_rna: 0 vs rest (4,444 vs 40,480 cells)
leiden_rna: 1 vs rest (6,163 vs 38,761 cells)
leiden_rna: 2 vs rest (1,811 vs 43,113 cells)
leiden_rna: 3 vs rest (1,281 vs 43,643 cells)
leiden_rna: 4 vs rest (12,440 vs 32,484 cells)
leiden_rna: 5 vs rest (522 vs 44,402 cells)
leiden_rna: 6 vs rest (1,399 vs 43,525 cells)
leiden_rna: 7 vs rest (6,773 vs 38,151 cells)
leiden_rna: 8 vs rest (3,255 vs 41,669 cells)
leiden_rna: 9 vs rest (4,232 vs 40,692 cells)
leiden_rna: 10 vs rest (2,119 vs 42,805 cells)
leiden_rna: 11 vs rest (17 vs 44,907 cells)
leiden_rna: 12 vs rest (468 vs 44,456 cells)
Computed expanded Wilcoxon payloads for: ['leiden_rna']


## Inspect The DE Payload Before Marker Selection

This table shows the top retained DE rows per category before the final `Features > Markers` cap is applied.

In [72]:
def de_payload_to_table(payload):
    rows = []
    for annotation, by_source in payload.items():
        for source, by_reference in by_source.items():
            if str(source).startswith("_"):
                continue
            for reference, result in by_reference.items():
                features = result.get("features") or []
                for rank, feature in enumerate(features, start=1):
                    rows.append({
                        "rank_in_de_payload": rank,
                        "annotation": annotation,
                        "category": source,
                        "reference": "rest" if reference == "__rest__" else reference,
                        "feature": feature,
                        "pval": (result.get("pvals") or [None] * len(features))[rank - 1],
                        "padj": (result.get("pvals_adj") or [None] * len(features))[rank - 1],
                        "log2fc": (result.get("log2foldchanges") or [None] * len(features))[rank - 1],
                        "score": (result.get("scores") or [None] * len(features))[rank - 1],
                        "pct_source": (result.get("pct_source") or [None] * len(features))[rank - 1],
                        "pct_reference": (result.get("pct_reference") or [None] * len(features))[rank - 1],
                        "n_source": result.get("n_source"),
                        "n_reference": result.get("n_reference"),
                    })
    return pd.DataFrame(rows)


de_table = de_payload_to_table(wilcoxon_de)
de_table.head(20)


,rank_in_de_payload,annotation,category,reference,feature,pval,padj,log2fc,score,pct_source,pct_reference,n_source,n_reference
0,1,leiden_rna,0,rest,DLK1,0.0,4.940656e-324,6.95640,79.5284,0.73155,0.005657,4444,40480
1,2,leiden_rna,0,rest,HS6ST2,0.0,4.940656e-324,6.55262,51.5508,0.47525,0.003780,4444,40480
2,3,leiden_rna,0,rest,DKK1,0.0,4.940656e-324,5.77640,51.8441,0.48177,0.008127,4444,40480
3,4,leiden_rna,0,rest,RELN,0.0,4.940656e-324,5.73168,80.7079,0.75158,0.012080,4444,40480
4,5,leiden_rna,0,rest,EPCAM,0.0,4.940656e-324,5.70519,90.4698,0.84248,0.015168,4444,40480
5,6,leiden_rna,0,rest,ANGPTL8,0.0,4.940656e-324,5.46027,58.7267,0.54838,0.010944,4444,40480
6,7,leiden_rna,0,rest,CLGN,0.0,4.940656e-324,5.32054,48.7749,0.45612,0.008943,4444,40480
7,8,leiden_rna,0,rest,PROM1,0.0,4.940656e-324,5.13311,40.2435,0.37714,0.007930,4444,40480
8,9,leiden_rna,0,rest,FGFR4,0.0,4.940656e-324,5.02305,55.5233,0.52273,0.012006,4444,40480
9,10,leiden_rna,0,rest,PEG10,0.0,4.940656e-324,4.85870,79.1493,0.74550,0.029348,4444,40480


## Recreate `marker_features_by_method_by_modality`

This is the final reduction that feeds `Features > Markers`. It uses the already-formatted DE payload and keeps only source-enriched significant markers.

In [73]:
def select_marker_features_from_de_payload(payload, padj_threshold, log2fc_threshold, limit_per_category=50):
    markers = {}
    rest_reference_key = "__rest__"
    for annotation_name, by_source in payload.items():
        annotation_markers = {}
        for source, by_reference in by_source.items():
            if str(source).startswith("_") or not isinstance(by_reference, dict):
                continue
            references = [(rest_reference_key, by_reference[rest_reference_key])] if rest_reference_key in by_reference else list(by_reference.items())
            ranked = {}
            for reference, result in references:
                if not isinstance(result, dict):
                    continue
                for feature, padj, log2fc in zip(result.get("features") or [], result.get("pvals_adj") or [], result.get("log2foldchanges") or []):
                    try:
                        padj_value = float(padj)
                        log2fc_value = float(log2fc)
                    except (TypeError, ValueError):
                        continue
                    if np.isfinite(padj_value) and np.isfinite(log2fc_value) and padj_value < float(padj_threshold) and log2fc_value >= float(log2fc_threshold):
                        score = (padj_value, -abs(log2fc_value))
                        feature = str(feature)
                        if feature not in ranked or score < ranked[feature]:
                            ranked[feature] = score
            if ranked:
                annotation_markers[str(source)] = [
                    feature
                    for feature, _score in sorted(ranked.items(), key=lambda item: (item[1][0], item[1][1], item[0]))[: int(limit_per_category)]
                ]
        if annotation_markers:
            markers[str(annotation_name)] = annotation_markers
    return markers


def marker_selection_table(payload, padj_threshold, log2fc_threshold, limit_per_category=50):
    rows = []
    selected = select_marker_features_from_de_payload(payload, padj_threshold, log2fc_threshold, limit_per_category)
    de = de_payload_to_table(payload)
    if de.empty:
        return de
    for annotation, by_category in selected.items():
        for category, features in by_category.items():
            subset = de[(de["annotation"] == annotation) & (de["category"] == category)].copy()
            for rank, feature in enumerate(features, start=1):
                match = subset[subset["feature"] == feature].head(1)
                if match.empty:
                    rows.append({"rank": rank, "annotation": annotation, "category": category, "feature": feature})
                else:
                    row = match.iloc[0].to_dict()
                    row["rank"] = rank
                    rows.append(row)
    columns = ["rank", "annotation", "category", "reference", "feature", "padj", "log2fc", "pval", "score", "pct_source", "pct_reference", "n_source", "n_reference"]
    return pd.DataFrame(rows).reindex(columns=columns)


wilcoxon_marker_features = select_marker_features_from_de_payload(
    wilcoxon_de,
    padj_threshold=wilcoxon_padj_cutoff,
    log2fc_threshold=wilcoxon_log2fc_cutoff,
    limit_per_category=marker_limit_per_category,
)
method_payload = {}
if wilcoxon_marker_features:
    method_payload["wilcoxon"] = wilcoxon_marker_features
marker_features_by_method_by_modality = {modality: method_payload}

marker_table = marker_selection_table(wilcoxon_de, wilcoxon_padj_cutoff, wilcoxon_log2fc_cutoff, marker_limit_per_category)
marker_table.head(20)


,rank,annotation,category,reference,feature,padj,log2fc,pval,score,pct_source,pct_reference,n_source,n_reference
0,1,leiden_rna,0,rest,DLK1,4.940656e-324,6.95640,0.0,79.5284,0.73155,0.005657,4444,40480
1,2,leiden_rna,0,rest,HS6ST2,4.940656e-324,6.55262,0.0,51.5508,0.47525,0.003780,4444,40480
2,3,leiden_rna,0,rest,DKK1,4.940656e-324,5.77640,0.0,51.8441,0.48177,0.008127,4444,40480
3,4,leiden_rna,0,rest,RELN,4.940656e-324,5.73168,0.0,80.7079,0.75158,0.012080,4444,40480
4,5,leiden_rna,0,rest,EPCAM,4.940656e-324,5.70519,0.0,90.4698,0.84248,0.015168,4444,40480
5,6,leiden_rna,0,rest,ANGPTL8,4.940656e-324,5.46027,0.0,58.7267,0.54838,0.010944,4444,40480
6,7,leiden_rna,0,rest,CLGN,4.940656e-324,5.32054,0.0,48.7749,0.45612,0.008943,4444,40480
7,8,leiden_rna,0,rest,PROM1,4.940656e-324,5.13311,0.0,40.2435,0.37714,0.007930,4444,40480
8,9,leiden_rna,0,rest,FGFR4,4.940656e-324,5.02305,0.0,55.5233,0.52273,0.012006,4444,40480
9,10,leiden_rna,0,rest,PEG10,4.940656e-324,4.85870,0.0,79.1493,0.74550,0.029348,4444,40480
